In [1]:
import torch
from pathlib import Path
from models import SmallCnn,resnet18_model
import numpy as np
import pandas as pd
from collections import defaultdict

from utils import (
run_one_epoch,
set_seed,
evaluate_model,
calculate_classification_metrics,
plot_confusion_matrices,
print_metrics,
extract_predictions_and_confidence
)
from torchvision.models import resnet18,ResNet18_Weights

from datasets import val_loader,val_loader_resnet,val_dataset_resnet
from sklearn.metrics import ConfusionMatrixDisplay,confusion_matrix
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


#### train baseline with CE,dropout=0.0

In [ ]:
# ==============================================
# use the best model we saved and get metrics.
# ==============================================
set_seed(42)
model_best = SmallCnn(
    num_classes=8,
    dropout=0
).to(device)
state_dict = torch.load("../results/saved/best_baseline_small_cnn_0.0.pt")
model_best.load_state_dict(state_dict)
model_best.eval()
best_val_metrics = run_one_epoch(
    model_best,
    val_loader,
    optimizer=None,
    device=device
)
print(best_val_metrics)

# Epoch 46/50
# Train Accuracy: 99.7%
# Val Accuracy:   83.2%
# Val Loss:       0.651


In [ ]:
y_true_base, y_pred_base = evaluate_model(
    model_best,
    val_loader,
    device=device
)

print(len(y_true_base))
print(len(y_pred_base))
print(y_true_base[:10])
print(y_pred_base[:10])

In [ ]:
classes = ['ambulance','autobus','kamyun','kamyunet','minibus','savari','taxi','vanet']

metrics_base = calculate_classification_metrics(y_true_base, y_pred_base)

print_metrics(metrics_base,classes)


In [ ]:
cm_base = confusion_matrix(y_true_base,y_pred_base)
print(cm_base)
cm_base.shape

In [ ]:
cm_normalized_base = cm_base / cm_base.sum(axis=1, keepdims=True)
print(cm_normalized_base)
# قطر اصلی ماتریس همان recall هر کلاس است

In [ ]:
output_dir = Path("../results/img")
output_dir.mkdir(parents=True, exist_ok=True)

cmd = ConfusionMatrixDisplay(
    confusion_matrix=cm_base,
    display_labels=classes
)
cmd.plot(xticks_rotation=45)
plt.title("CNN Baseline - Confusion Matrix")
plt.tight_layout()
plt.savefig(
    output_dir / "SmallCnn_base_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

In [ ]:
cmd = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized_base,
    display_labels=classes
)
cmd.plot(xticks_rotation=45)
plt.title("CNN Baseline - Normalized Confusion Matrix")
plt.tight_layout()
plt.savefig(
    output_dir / "SmallCnn_base_confusion_matrix_normalized.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

SmallCnn Augmented

In [ ]:
model_aug_best = SmallCnn(
    num_classes=8,
    dropout=0
).to(device)

state_dict = torch.load(
    "../results/saved/best_augmented_small_cnn.pt",
    map_location=device
)

model_aug_best.load_state_dict(state_dict)

In [ ]:
y_true_aug, y_pred_aug = evaluate_model(
    model_aug_best,
    val_loader,
    device=device
)

In [ ]:
metrics_aug = calculate_classification_metrics(y_true_aug, y_pred_aug)
print_metrics(metrics_aug,classes)


In [ ]:
cm_aug = confusion_matrix(y_true_aug,y_pred_aug)
print(cm_aug)
cm_aug.shape

In [ ]:
cm_normalized_aug = cm_aug / cm_aug.sum(axis=1, keepdims=True)
print(cm_normalized_aug)


In [ ]:
output_dir = Path("../results/img")
output_dir.mkdir(parents=True, exist_ok=True)

cmd = ConfusionMatrixDisplay(
    confusion_matrix=cm_aug,
    display_labels=classes
)
cmd.plot(xticks_rotation=45)
plt.title("CNN augmented - Confusion Matrix")
plt.tight_layout()
plt.savefig(
    output_dir / "SmallCnn_aug_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

In [ ]:
output_dir = Path("../results/img")
output_dir.mkdir(parents=True, exist_ok=True)

cmd = ConfusionMatrixDisplay(
    confusion_matrix=cm_normalized_aug,
    display_labels=classes
)
cmd.plot(xticks_rotation=45)
plt.title("CNN augmented - Normalize Confusion Matrix")
plt.tight_layout()
plt.savefig(
    output_dir / "SmallCnn_aug_confusion_matrix_normalized.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()


### small cnn with dropout [0.3, 0.5]


In [ ]:
def evaluate_checkpoint(checkpoint_path, dropout, val_loader, device):
    model = SmallCnn(
        num_classes=8,
        dropout=dropout
    ).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)

    y_true , y_pred = evaluate_model(
        model,
        val_loader,
        device=device
    )
    cm_drop, cm_normalized_drop = plot_confusion_matrices(
    y_true=y_true,
    y_pred=y_pred,
    classes=classes,
    experiment_name=f"small_cnn_dropout_{dropout}",
    output_dir="../results/img"
)
    metrics = calculate_classification_metrics(y_true,y_pred)

    return metrics,cm_drop,cm_normalized_drop

In [ ]:
checkpoint_dir = Path("../results/saved")

dropout_experiments = {
    "dropout_0.0": {
        "dropout": 0.0,
        "checkpoint": checkpoint_dir / "best_baseline_small_cnn_0.0.pt"
    },
    "dropout_0.3": {
        "dropout": 0.3,
        "checkpoint": checkpoint_dir / "best_baseline_small_cnn_0.3.pt"
    },
    "dropout_0.5": {
        "dropout": 0.5,
        "checkpoint": checkpoint_dir / "best_baseline_small_cnn_0.5.pt"
    }
}

In [ ]:
dropout_metrics = {}
cm_drop_dict = {}
for name, config in dropout_experiments.items():

    metrics,cm_drop,cm_normalized_drop = evaluate_checkpoint(
        checkpoint_path=config["checkpoint"],
        dropout=config["dropout"],
        val_loader=val_loader,
        device=device
    )

    dropout_metrics[name] = metrics
    cm_drop_dict[name] = [cm_drop,cm_normalized_drop]

In [ ]:
print("\n=== Dropout Comparison ===")

print(
    f"{'Dropout':<10}"
    f"{'Precision':<12}"
    f"{'Recall':<12}"
    f"{'F1':<12}"
)

for name, metrics in dropout_metrics.items():

    dropout = name.replace("dropout_", "")

    print(
        f"{dropout:<10}"
        f"{metrics['precision_macro']:<12.4f}"
        f"{metrics['recall_macro']:<12.4f}"
        f"{metrics['f1_macro']:<12.5f}"
    )

In [ ]:
for name, metrics in dropout_metrics.items():

    print(f"\n=== {name} - Per Class ===")
    print_metrics(metrics,classes)

### avgpool

In [ ]:
model_avg_best = SmallCnn(
    num_classes=8,
    dropout=0.0,
    pooling="avg"
).to(device)

state_dict = torch.load(
    "../results/saved/best_small_cnn_avg_pool.pt",
    map_location=device
)

model_avg_best.load_state_dict(state_dict)
model_avg_best

In [ ]:
model_avg_best.eval()
best_avg_metrics = run_one_epoch(
    model_avg_best,
    val_loader,
    optimizer=None,
    device=device
)
print(best_avg_metrics)


In [ ]:
y_true_avg, y_pred_avg = evaluate_model(
    model_avg_best,
    val_loader,
    device=device
)
avg_metrics = calculate_classification_metrics(
    y_true=y_true_avg,
    y_pred=y_pred_avg,
)


In [ ]:
print_metrics(avg_metrics,classes)


In [ ]:
cm_avg, cm_normalized_avg = plot_confusion_matrices(
    y_true=y_true_avg,
    y_pred=y_pred_avg,
    classes=classes,
    experiment_name="small_cnn_avg_pool",
    output_dir="../results/img"
)

### weight decay

In [ ]:
model_weight_decay_best = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max"
).to(device)

state_dict = torch.load(
    "../results/saved/best_weight_decay_small_cnn_0.0001.pt",
    map_location=device
)

model_weight_decay_best.load_state_dict(state_dict)


model_weight_decay_best.eval()
best_weight_decay_metrics = run_one_epoch(
    model_weight_decay_best,
    val_loader,
    optimizer=None,
    device=device
)
print(best_weight_decay_metrics)

In [ ]:
y_true_wd , y_pred_wd = evaluate_model(
    model_weight_decay_best,
    val_loader,
    device=device
)
len(y_true_wd)

In [ ]:
wd_metrics = calculate_classification_metrics(
    y_true_wd,
    y_pred_wd
)
print_metrics(wd_metrics, classes)

In [ ]:
cm_wd, cm_normalized_wd = plot_confusion_matrices(
    y_true=y_true_wd,
    y_pred=y_pred_wd,
    classes=classes,
    experiment_name="small_cnn_weight_decay_0.0001",
    output_dir="../results/img"
)

#### StepLR

In [ ]:
model_lr = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max"
).to(device)
state_dict = torch.load("../results/saved/best_small_cnn_step_lr.pt")
model_lr.load_state_dict(state_dict)
model_lr.eval()
best_lr_metrics = run_one_epoch(
    model_lr,
    val_loader,
    device=device
)
print(best_lr_metrics)

In [ ]:
y_true_lr, y_pred_lr = evaluate_model(
    model_lr,
    val_loader,
    device=device
)
len(y_true_lr)

In [ ]:
lr_metrics = calculate_classification_metrics(
    y_true_lr,
    y_pred_lr
)
print_metrics(lr_metrics, classes)

In [ ]:
cm_lr, cm_normalized_lr = plot_confusion_matrices(
    y_true=y_true_lr,
    y_pred=y_pred_lr,
    classes=classes,
    experiment_name="small_cnn_step_lr",
    output_dir="../results/img"
)

### ReduceLROnPlateau

In [ ]:
model_lrp = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max",
).to(device)
state_dict = torch.load("../results/saved/best_reduce_lr_small_cnn.pt")
model_lrp.load_state_dict(state_dict)
model_lrp.eval()
best_lrp_metrics = run_one_epoch(
    model_lrp,
    val_loader,
    device=device,
)
best_lrp_metrics

In [ ]:
y_true_rlr, y_pred_rlr = evaluate_model(
    model_lrp,
    val_loader,
    device=device
)
len(y_true_rlr)

In [ ]:
rlr_metrics = calculate_classification_metrics(
    y_true_rlr,
    y_pred_rlr
)
print_metrics(rlr_metrics, classes)


In [ ]:
cm_rlr, cm_normalized_rlr = plot_confusion_matrices(
    y_true=y_true_rlr,
    y_pred=y_pred_rlr,
    classes=classes,
    experiment_name="best_reduce_lr_small_cnn",
    output_dir="../results/img"
)

### Imbalanced datasets

In [ ]:
model_standard = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max",
).to(device)
state_dict = torch.load("../results/saved/train_loader_standard_imbalanced.pt")
model_standard.load_state_dict(state_dict)
model_standard.eval()
best_standard_metrics = run_one_epoch(
    model_standard,
    val_loader,
    device=device,
)
best_standard_metrics

In [ ]:
y_true_standard, y_pred_standard = evaluate_model(
    model_standard,
    val_loader,
    device=device
)
len(y_true_standard)

In [ ]:
standard_metrics = calculate_classification_metrics(
    y_true_standard,
    y_pred_standard
)
print_metrics(standard_metrics, classes)

In [ ]:
cm_standard, cm_normalized_standard = plot_confusion_matrices(
    y_true=y_true_standard,
    y_pred=y_pred_standard,
    classes=classes,
    experiment_name="best_standard_small_cnn",
    output_dir="../results/img"
)

In [ ]:
model_balanced = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max",
).to(device)
state_dict = torch.load("../results/saved/train_balanced_imbalanced.pt")
model_balanced.load_state_dict(state_dict)
model_balanced.eval()
best_balanced_metrics = run_one_epoch(
    model_balanced,
    val_loader,
    device=device,
)
best_balanced_metrics

In [ ]:
y_true_balanced, y_pred_balanced = evaluate_model(
    model_balanced,
    val_loader,
    device=device
)
len(y_true_balanced)

In [ ]:
balanced_metrics = calculate_classification_metrics(
    y_true_balanced,
    y_pred_balanced
)
print_metrics(balanced_metrics, classes)


In [ ]:
cm_balanced, cm_normalized_balanced = plot_confusion_matrices(
    y_true=y_true_balanced,
    y_pred=y_pred_balanced,
    classes=classes,
    experiment_name="best_balanced_small_cnn",
    output_dir="../results/img"
)

## train baseline with BCE

In [ ]:
model_bce_best = SmallCnn(
    num_classes=8,
    dropout=0,
    pooling="max"
).to(device)

state_dict = torch.load(
    "../results/saved/best_bce_small_cnn.pt",
    map_location=device
)

model_bce_best.load_state_dict(state_dict)

y_true_bce, y_pred_bce = evaluate_model(
    model_bce_best,
    val_loader,
    device=device
)

metrics_bce = calculate_classification_metrics(
    y_true_bce,
    y_pred_bce
)
print_metrics(metrics_bce,classes)

In [ ]:
cm_bce, cm_normalized_bce = plot_confusion_matrices(
    y_true=y_true_bce,
    y_pred=y_pred_bce,
    classes=classes,
    experiment_name="best_bce_small_cnn",
    output_dir="../results/img"
)

#### predictions_and_confidence for CE and BCE

In [ ]:
y_true_ce, y_pred_ce, conf_ce, scores_ce = extract_predictions_and_confidence(
    model_best,
    val_loader,
    device=device,
    loss_type="ce"
)
y_true_bce, y_pred_bce, conf_bce, scores_bce = (
    extract_predictions_and_confidence(
        model_bce_best,
        val_loader,
        device,
        loss_type="bce"
    )
)


In [ ]:
print("CE")
print("mean  :", np.mean(conf_ce))
print("median:", np.median(conf_ce))
print("min   :", np.min(conf_ce))
print("max   :", np.max(conf_ce))

print("\nBCE")
print("mean  :", np.mean(conf_bce))
print("median:", np.median(conf_bce))
print("min   :", np.min(conf_bce))
print("max   :", np.max(conf_bce))
# مدل CE معمولاً predictionهایش را با اطمینان بیشتری بیان می‌کند.

In [ ]:
correct_ce = np.array(conf_ce)[
    np.array(y_true_ce) == np.array(y_pred_ce)
]

wrong_ce = np.array(conf_ce)[
    np.array(y_true_ce) != np.array(y_pred_ce)
]

correct_bce = np.array(conf_bce)[
    np.array(y_true_bce) == np.array(y_pred_bce)
]

wrong_bce = np.array(conf_bce)[
    np.array(y_true_bce) != np.array(y_pred_bce)
]
print("CE")
print("correct mean:", correct_ce.mean())
print("wrong mean  :", wrong_ce.mean())

print("\nBCE")
print("correct mean:", correct_bce.mean())
print("wrong mean  :", wrong_bce.mean())
# CE حتی وقتی اشتباه می‌کند، confidence آن هنوز نسبتاً بالا است.

<div dir=rtl>BCE از نظر classification performance ضعیف‌تر بود، اما در این آزمایش روی خطاهایش کمتر با اطمینان بالا عمل کرده است.</div>

#### ResNet18 pretrained

In [ ]:
best_resnet_model = resnet18_model(
    num_classes=8
).to(device)
state_dict = torch.load(
    "../results/saved/resnet18_pretrained.pt",
    map_location=device
)
best_resnet_model.load_state_dict(state_dict)
best_resnet_model.eval()
y_true_res_pre , y_pred_res_pre = evaluate_model(
    best_resnet_model,
    val_loader_resnet,
    device=device
)
res_pre_metrics = calculate_classification_metrics(
    y_true_res_pre,
    y_pred_res_pre
)

In [ ]:
print_metrics(res_pre_metrics,classes)

In [ ]:
cm_res_pre , cm_res_normalized_pre = plot_confusion_matrices(
    y_true_res_pre,
    y_pred_res_pre,
    classes=classes,
    experiment_name="resnet18_pretrained",
    output_dir="../results/img",
)

#### ResNet18 fine-tuned

In [ ]:
model_resnet_ft = resnet18_model(
    num_classes=8
).to(device)
for param in model_resnet_ft.parameters():
    param.requires_grad = False

for param in model_resnet_ft.layer4.parameters():
    param.requires_grad = True

for param in model_resnet_ft.fc.parameters():
    param.requires_grad = True

model_resnet_ft.load_state_dict(
    torch.load(
        "../results/saved/best_resnet18_ft.pt",
        map_location=device
    )
)
model_resnet_ft.eval()

y_true_res_ft , y_pred_res_ft = evaluate_model(
    model_resnet_ft,
    val_loader_resnet,
    device=device
)
res_ft_metrics = calculate_classification_metrics(
    y_true_res_ft,
    y_pred_res_ft
)

In [ ]:
print_metrics(res_ft_metrics,classes)

In [ ]:
cm_res_ft , cm_res_normalized_ft = plot_confusion_matrices(
    y_true_res_ft,
    y_pred_res_ft,
    classes=classes,
    experiment_name="resnet18_ft",
    output_dir="../results/img",
)

In [ ]:
targets, predictions, confidences, scores = extract_predictions_and_confidence(
    model=model_resnet_ft,
    loader=val_loader_resnet,
    device=device,
    loss_type="ce"
)
misclassified_indices = [
    i for i, (true_label, pred_label) in enumerate(zip(targets, predictions)) if true_label != pred_label
]
print(f"Total misclassified: {len(misclassified_indices)}")
for idx in misclassified_indices[:12]:
    image ,label = val_dataset_resnet[idx]
    print(
        f"Index: {idx} | "
        f"True: {val_dataset_resnet.classes[targets[idx]]} | "
        f"Pred: {val_dataset_resnet.classes[predictions[idx]]} | "
        f"Confidence: {confidences[idx]:.2%} | "
        f"Path: {val_dataset_resnet.samples[idx][0]}"
    )

In [ ]:
weights = ResNet18_Weights.DEFAULT
mean = torch.tensor(weights.transforms().mean).view(3, 1, 1)
std = torch.tensor(weights.transforms().std).view(3, 1, 1)

fig, axes = plt.subplots(4, 3, figsize=(16, 12))

for ax, idx in zip(axes.flat, misclassified_indices[:12]):

    image, _ = val_dataset_resnet[idx]
    image = image * std + mean
    image = image.clamp(0, 1)

    # CHW -> HWC
    image = image.permute(1, 2, 0)

    ax.imshow(image)
    ax.set_title(
        f"True: {val_dataset_resnet.classes[targets[idx]]}\n"
        f"Pred: {val_dataset_resnet.classes[predictions[idx]]}\n"
        f"Conf: {confidences[idx]:.2%}"
    )

    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "../results/img/resnet_ft_misclassified_12.png",
    bbox_inches="tight",
    dpi=300,
    format="png",
)
plt.show()


#### Mutual Confusion

In [ ]:
pair_confusions = []

for i in range(len(classes)):
    for j in range(i + 1, len(classes)):

        confusion_ij = cm_res_normalized_ft[i, j]
        confusion_ji = cm_res_normalized_ft[j, i]

        pair_confusion = confusion_ij + confusion_ji

        pair_confusions.append({
            "Class 1": classes[i],
            "Class 2": classes[j],
            "Class1 -> Class2": confusion_ij,
            "Class2 -> Class1": confusion_ji,
            "Mutual Confusion": pair_confusion
        })

pair_confusions_df = pd.DataFrame(pair_confusions)

pair_confusions_df = pair_confusions_df.sort_values(
    "Mutual Confusion",
    ascending=False
).reset_index(drop=True)

pair_confusions_df

#### check confidence and propose viewing by human operator

In [ ]:
classes = val_loader_resnet.dataset.classes

results_df = pd.DataFrame({
    "true_label": [classes[i] for i in targets],
    "predicted_label": [classes[i] for i in predictions],
    "confidence": confidences,
})
results_df

In [ ]:
thresholds = np.arange(0.70, 0.96, 0.05)
predictions = np.array(predictions)
targets = np.array(targets)
threshold_results = []
for threshold in thresholds:
    needs_review = confidences < threshold
    auto_accept = ~needs_review
    total = len(targets)
    review_count = needs_review.sum()
    auto_count = auto_accept.sum()
    # print(total, review_count, auto_count)
    review_rate = review_count / total
    coverage = auto_count / total
    overall_accuracy = np.mean(predictions == targets)

    if auto_count > 0:
        auto_accuracy = (
            predictions[auto_accept] == targets[auto_accept]
        ).mean()

        auto_errors = (
            predictions[auto_accept] != targets[auto_accept]
        ).sum()

        auto_error_rate = auto_errors / auto_count

    else:
        auto_accuracy = np.nan
        auto_errors = 0
        auto_error_rate = np.nan

    threshold_results.append({
        "threshold": threshold,
        "review_count": review_count,
        "review_rate": review_rate,
        "coverage": coverage,
        "auto_accuracy": auto_accuracy,
        "auto_error_rate": auto_error_rate,
    })


threshold_df = pd.DataFrame(threshold_results)
threshold_display = threshold_df.copy()

threshold_display["review_rate"] = (threshold_display["review_rate"] * 100).round(2)

threshold_display["coverage"] = (threshold_display["coverage"] * 100).round(2)

threshold_display["auto_accuracy"] = (threshold_display["auto_accuracy"] * 100).round(2)

threshold_display["auto_error_rate"] = (threshold_display["auto_error_rate"] * 100).round(2)

threshold_display

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_df["threshold"],
    threshold_df["review_rate"] * 100,
    marker="o",
    label="Review Rate"
)

plt.plot(
    threshold_df["threshold"],
    threshold_df["auto_error_rate"] * 100,
    marker="o",
    label="Auto Error Rate"
)

plt.xlabel("Confidence Threshold")
plt.ylabel("Percentage")
plt.title("Confidence Threshold Analysis on Validation Set")
plt.legend()
plt.grid(True)
plt.savefig(
    "../results/img/resnet_ft_threshold_analysis.png",
    bbox_inches="tight",
    dpi=300,
)
plt.show()

In [ ]:
threshold_error_results = []

for threshold in thresholds:
    auto_accept = confidences >= threshold

    auto_count = auto_accept.sum()

    wrong_auto = (
        (predictions != targets) &
        auto_accept
    )

    wrong_auto_count = wrong_auto.sum()

    threshold_error_results.append({
        "threshold": threshold,
        "auto_count": auto_count,
        "wrong_auto_count": wrong_auto_count,
        "wrong_auto_rate": (
            wrong_auto_count / auto_count * 100
            if auto_count > 0 else np.nan
        )
    })

threshold_error_df = pd.DataFrame(threshold_error_results)

print(threshold_error_df)

In [ ]:
wrong_predictions_df = results_df[
    results_df["true_label"] != results_df["predicted_label"]
].sort_values(
    by="confidence",
    ascending=False
)
print(wrong_predictions_df)

In [ ]:
threshold_df[
    [
        "threshold",
        "coverage",
        "review_rate",
        "auto_accuracy",
        "auto_error_rate"
    ]
].round(2)